# Запуск pipeline на Google Colab

1. Монтируем Google Drive
2. Настраиваем Kaggle API токен
3. Скачиваем сырые данные с Kaggle (если ещё нет)
4. Копируем код проекта, ставим зависимости
5. Распаковываем архивы
6. Запускаем pipeline

## 0. Перед запуском ноутбука переметите [данные](https://drive.google.com/drive/folders/14oi8Z5y74okkTlFBDj7IO_hnoJTV8jqw?usp=sharing) на свой Google Drive

Запускали командой через [расширение для VS Code для работы с Google Colab](https://marketplace.visualstudio.com/items?itemName=Google.colab).

**После НЕобязательно делать токен, так как все данные будут скачены уже.**

Но это раздел с токеном решили оставить на всякий.

In [1]:
# 1. Монтируем Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 2. Настраиваем Kaggle API токен (читаем с Drive, НЕ хардкодим)
import os, shutil

KAGGLE_JSON_SRC = "/content/drive/MyDrive/favorita/kaggle.json"
KAGGLE_DIR = os.path.expanduser("~/.kaggle")

if not os.path.exists(KAGGLE_JSON_SRC):
    raise FileNotFoundError(
        f"Не найден {KAGGLE_JSON_SRC}\n"
        "Скачай kaggle.json с https://www.kaggle.com/settings → API → Create New Token\n"
        "и положи файл на Google Drive в My Drive/favorita/kaggle.json"
    )

os.makedirs(KAGGLE_DIR, exist_ok=True)
shutil.copy(KAGGLE_JSON_SRC, os.path.join(KAGGLE_DIR, "kaggle.json"))
os.chmod(os.path.join(KAGGLE_DIR, "kaggle.json"), 0o600)
print("Kaggle API токен установлен")

Kaggle API токен установлен


In [2]:
import os, shutil

# 3. Скачиваем данные с Kaggle (если архивов ещё нет на Drive)
RAW_DIR = "/content/drive/MyDrive/favorita/raw_7z"
os.makedirs(RAW_DIR, exist_ok=True)

existing = [f for f in os.listdir(RAW_DIR) if f.endswith(".7z")]
if len(existing) >= 6:
    print(f"Архивы уже на Drive ({len(existing)} файлов), пропускаем загрузку")
else:
    print("Скачиваем датасет с Kaggle...")
    !pip install -q kaggle
    !kaggle competitions download -c favorita-grocery-sales-forecasting -p /tmp/favorita_raw

    import glob
    for f in glob.glob("/tmp/favorita_raw/*.7z"):
        dest = os.path.join(RAW_DIR, os.path.basename(f))
        if not os.path.exists(dest):
            shutil.move(f, dest)
            print(f"  → {os.path.basename(f)}")

    print(f"\nГотово! Файлов на Drive: {len(os.listdir(RAW_DIR))}")

!ls -lh {RAW_DIR}

Архивы уже на Drive (8 файлов), пропускаем загрузку
total 458M
-rw------- 1 root root 1.9K Dec 11  2019 holidays_events.csv.7z
-rw------- 1 root root  14K Dec 11  2019 items.csv.7z
-rw------- 1 root root 3.7K Dec 11  2019 oil.csv.7z
-rw------- 1 root root 651K Dec 11  2019 sample_submission.csv.7z
-rw------- 1 root root  648 Dec 11  2019 stores.csv.7z
-rw------- 1 root root 4.7M Dec 11  2019 test.csv.7z
-rw------- 1 root root 453M Dec 11  2019 train.csv.7z
-rw------- 1 root root 215K Dec 11  2019 transactions.csv.7z


In [3]:
# 4. Копируем код проекта с Drive на Colab + ставим зависимости
PROJECT_DIR = "/content/drive/MyDrive/favorita/project"

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Не найден {PROJECT_DIR}\n"
        "Загрузи папку проекта (src/, run_experiment.py, requirements.txt) "
        "на Google Drive в My Drive/favorita/project/"
    )

!cp -r {PROJECT_DIR} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!ls -la

/content/project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 26.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 67.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 7.3 MB/s eta 0:00:00
total 44
drwx------ 4 root root  4096 Mar 18 18:30 .
drwxr-xr-x 1 root root  4096 Mar 18 18:29 ..
-rw------- 1 root root   134 Mar 18 18:29 requirements.txt
drwx------ 2 root root  4096 Mar 18 18:30 results
-rw------- 1 root root  7435 Mar 18 18:29 run_colab.ipynb
-rw------- 1 root root 12369 Mar 18 18:29 run_experiment.py

In [4]:
# 5. Распаковка 7z → CSV (если ещё не распакованы)
CSV_DIR = "/content/drive/MyDrive/favorita/csv"

if not os.path.exists(os.path.join(CSV_DIR, "train.csv")):
    from src.data.extract import extract_archives
    extract_archives(RAW_DIR, CSV_DIR)
else:
    print("CSV уже распакованы")

!ls -lh {CSV_DIR}

CSV уже распакованы
total 4.9G
-rw------- 1 root root  22K Mar 17 14:28 holidays_events.csv
-rw------- 1 root root 100K Mar 17 14:28 items.csv
-rw------- 1 root root  21K Mar 17 14:28 oil.csv
-rw------- 1 root root  39M Mar 17 14:28 sample_submission.csv
-rw------- 1 root root 1.4K Mar 17 14:28 stores.csv
-rw------- 1 root root 121M Mar 17 14:28 test.csv
-rw------- 1 root root 4.7G Mar 17 14:34 train.csv
-rw------- 1 root root 1.5M Mar 17 14:34 transactions.csv


In [ ]:
# 6. Запуск pipeline
# Варианты:
#   --skip-fe           загрузить готовые фичи из parquet (быстрее)
#   --skip-baselines    пропустить статистические бейзлайны

# Быстрый запуск: только CatBoost + NN на кэшированных фичах
# !python run_experiment.py --skip-fe --skip-baselines

# Полный запуск с бейзлайнами (раскомментируй):
# !python run_experiment.py --skip-fe

# Полный запуск с FE с нуля (раскомментируй):
# !python run_experiment.py

Рекомендуется запусускать именно через эту команду, так как фичи лежат на Google Drive.

In [5]:
!python run_experiment.py --skip-fe

Step 1: Loading data
  train: (125497040, 6), test: (3370464, 5)

Step 2: Preparing data
  train_work: (38824824, 6), test_work: (3370464, 5)

Step 3: Feature Engineering
  Loading pre-computed features from parquet...
  Shapes: train=(10532700, 77), valid=(3370464, 77), test=(3370464, 77)

Step 4: Statistical Baselines
           model  NWRMSLE    WMAPE     BIAS
      AutoETS(7) 0.737961 0.770429 0.791629
    AutoTheta(7) 0.742213 0.767786 0.766521
SeasonalNaive(7) 0.755346 0.756975 0.410766
           Naive 0.763550 0.890571 1.377848

Step 5: CatBoost
  Features: 73, X_tr: (10532700, 73), X_va: (3370464, 73)
0:	learn: 1.0312684	test: 1.0323915	best: 1.0323915 (0)	total: 307ms	remaining: 12m 46s
200:	learn: 0.5383925	test: 0.6080572	best: 0.6080572 (200)	total: 1m 3s	remaining: 12m 2s
400:	learn: 0.5270847	test: 0.5732265	best: 0.5732265 (400)	total: 2m 7s	remaining: 11m 6s
600:	learn: 0.5215827	test: 0.5661672	best: 0.5660282 (592)	total: 3m 10s	remaining: 10m 2s
800:	learn: 0.517574